In [9]:
import os
import sys
import re
import time

project_root = "/root/work/tenset"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")
    

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build:" + os.environ.get("LD_LIBRARY_PATH", "")

1. Total 저장

In [ ]:
from glob import glob
import pandas as pd


dir_name = "/root/work/tenset/scripts/pre_experiments/model_myself/result_xgb/([[]0c9a5ba46ffc5e1a9e5641018527117f,4,16,16,112,1,1,112,672,1,1,1,672,4,16,16,672[]],cuda)"

csv_dir = glob(f"{dir_name}/**/*.csv", recursive=True)
csvs = []
for csv in csv_dir:
    if "avg" in csv or "total" in csv or "sampling" in csv or "prev" in csv:
       continue
    csvs.append(csv) 

dfs = []
for p in csvs:
    sub_df = pd.read_csv(p)

    if "rank_warmup_epochs" not in sub_df.columns:
        sub_df["rank_warmup_epochs"] = 0
    if "measure_size" not in sub_df.columns:
        sub_df["measure_size"] = 64
    if "scratch" not in sub_df.columns:
        sub_df["scratch"] = False
    if "encoder_freeze" not in sub_df.columns:
        sub_df["encoder_freeze"] = False
    
    dfs.append(sub_df)

    
df_total = pd.concat(dfs, ignore_index=True)

# measure_size 컬럼을 맨 앞으로 이동
cols = df_total.columns.tolist()
df_total = df_total[["measure_size"] + [c for c in cols if c != "measure_size"]]

df_total.to_csv(f"{os.path.dirname(csv_dir[0])}/xgb_extent_total.csv", index=True)
df_total

,measure_size,phase,used_time,train_size,top-{top_k},val_reg_r2,val_rank_r2,sampling_seed,rank_warmup_epochs,scratch,encoder_freeze,top-1
0,64,2,3.62,128,0.0,"[0.4806, 0.4979]",[],2000,0,False,False,NaN
1,64,7,8.70,448,0.0,"[0.4699, 0.5804, 0.5996, 0.556, 0.5277, 0.5244...",[],2001,0,False,False,NaN
2,64,8,10.23,512,0.0,"[0.4089, 0.5134, 0.4868, 0.4575, 0.4236, 0.482...",[],2002,0,False,False,NaN
3,64,6,7.12,384,0.0,"[0.4695, 0.542, 0.5432, 0.5655, 0.6208, 0.6233]",[],2003,0,False,False,NaN
4,64,6,7.19,384,0.0,"[0.403, 0.4552, 0.4697, 0.5477, 0.4957, 0.5281]",[],2004,0,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
75,64,2,3.44,128,NaN,"[0.4461, 0.4321]",[],2018,0,False,False,0.0
76,64,2,3.44,128,NaN,"[0.4467, 0.5333]",[],2018,0,False,False,0.0
77,64,1,2.96,64,NaN,[0.4342],[],2019,0,False,False,0.0
78,64,1,2.95,64,NaN,[0.4338],[],2019,0,False,False,0.0


2. Total avg 저장

In [15]:
agg_kwargs = {
    "phase": ("phase", "mean"),
    "train_size": ("train_size", "mean"),
    "used_time": ("used_time", "mean"),
    "val_reg_r2": ("val_reg_r2", "first"),
    "seed_n": ("sampling_seed", "nunique"),
    "sampling_seed": ("sampling_seed", list),
}

topk_col = f"top-1"
if topk_col in df_total.columns:
    agg_kwargs[topk_col] = (topk_col, "mean")

group_cols = [
    "measure_size",
    # "scratch",
    # "encoder_freeze",
    # "encoder_lr",
    # "cost_predictor_lr",
    # "rank_warmup_epochs",
    # "weights",
    # "uncertainty_topk",
    # "grad_num",
    # "rand_num",
]

df_total_avg = (
    df_total
    .groupby(group_cols, as_index=False)
    .agg(**agg_kwargs)
)
df_total_avg.to_csv(f"{os.path.dirname(csv_dir[0])}/xgb_extent_total_avg.csv", index=False)
df_total_avg

,measure_size,phase,train_size,used_time,val_reg_r2,seed_n,sampling_seed,top-1
0,64,4.45,284.8,6.097125,"[0.4806, 0.4979]",20,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200...",0.066667


선택 파일 Avg 저장

In [11]:
def save_avg_csv(filename, top_k):
    df_results = pd.read_csv(filename)
    group_cols = [
        "measure_size",
        # "scratch",
        # "encoder_freeze",
        # "encoder_lr",
        # "cost_predictor_lr",
        # "rank_warmup_epochs",
        # "weights",
        # "uncertainty_topk",
        # "grad_num",
        # "rand_num",
    ]

    df_avg = (
        df_results
        .groupby(group_cols, as_index=False)
        .agg(
            phase=("phase", "mean"),
            train_size=("train_size", "mean"),
            used_time=("used_time", "mean"),
            # **{f"top-{top_k}": (f"top-{top_k}", "mean")},
            val_reg_r2=("val_reg_r2", "first"),
            val_rank_r2=("val_rank_r2", "first"),
            seed_n=("sampling_seed", "nunique"),
            sampling_seed=("sampling_seed", list),
        )
    )

    df_avg.to_csv(filename.replace(".csv", "_avg.csv"), index=False)
    df_avg


save_avg_csv("/root/work/tenset/scripts/pre_experiments/model_myself/result_xgb/([8c674f26f66543069d1e1c56cda249f9,4,60,60,256,1,1,256,512,1,1,1,512,4,30,30,512],cuda).json/xgb_search_1222_1548.csv", 1)

두 csv 합치기

In [ ]:
import pandas as pd


# CSV 파일 읽기
csv1 = pd.read_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2013.csv')
csv2 = pd.read_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv')

# 두 CSV 합치기
merged_csv = pd.concat([csv1, csv2])

# 합친 CSV 저장
merged_csv.to_csv('/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_hyper_total.csv', index=False)
save_avg_csv(merged_csv, '/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_hyper_total.csv', top_k=1)

In [13]:
h = pd.read_csv("/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv")

h["train_size"] = h["train_size"] + 16
h.to_csv(
    "/root/work/tenset/scripts/pre_experiments/model_myself/result/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json/hyper/vae_extent_1220_2135.csv",
    index=False
)
